# Titanic 存活预测 —— 建模与评估

在 EDA 基础上完成：特征编码 → 训练 Logistic 回归 / 随机森林 → 交叉验证 → 生成提交文件。

运行前提：先运行 `01_eda.ipynb`，并确保 `data/train.csv`、`data/test.csv` 已就位。

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
from data_utils import load_data, impute_age, build_features
from model_utils import (
    encode_categorical, train_logistic, train_random_forest,
    evaluate, save_submission,
)

train, test = load_data()

## 1. 数据预处理（训练集 + 测试集统一处理）

In [ ]:
def preprocess(df):
    df = impute_age(df)
    df = build_features(df)
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    return df

train = preprocess(train)
test = preprocess(test)

# 选择进入模型的特征
feature_columns = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                   'Embarked', 'FamilySize', 'IsAlone', 'Title']

train = encode_categorical(train, ['Sex', 'Embarked', 'Title'])
test = encode_categorical(test, ['Sex', 'Embarked', 'Title'])

X = train[feature_columns]
y = train['Survived']
print('特征矩阵形状:', X.shape)

## 2. Logistic 回归

In [ ]:
lr = train_logistic(X, y)
cv_scores, conf_matrix, report = evaluate(lr, X, y)
print('交叉验证准确率:', cv_scores.round(4))
print('均值: %.4f (±%.4f)' % (cv_scores.mean(), cv_scores.std()))
print('\n混淆矩阵:\n', conf_matrix)
print('\n分类报告:\n', report)

## 3. 随机森林（通常优于单棵 Logistic）

In [ ]:
rf = train_random_forest(X, y)
rf_scores, rf_conf, rf_report = evaluate(rf, X, y)
print('随机森林交叉验证准确率均值: %.4f' % rf_scores.mean())
print('\n混淆矩阵:\n', rf_conf)

## 4. 生成提交文件

用表现更好的模型（可自行切换 lr / rf）生成 Kaggle 提交文件。

In [ ]:
best_model = rf  # 若 Logistic 更好则改为 lr
submission = save_submission(best_model, test, feature_columns)
submission.head()